In [ ]:
import numpy as np  # Import numpy for numerical operations and arrays
import pandas as pd  # Import pandas for data manipulation
from sklearn.model_selection import train_test_split  # Import function to split data into train/test sets
from sklearn.preprocessing import OneHotEncoder  # Import encoder for categorical variables
from sklearn.compose import ColumnTransformer  # Import tool to apply different preprocessing to different columns
from sklearn.linear_model import SGDClassifier, LogisticRegression  # Import linear classifiers: SGD and Logistic Regression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score  # Import evaluation metrics
from sklearn.datasets import make_classification  # Import function to generate synthetic classification datasets

# Load CSV file
df = pd.read_csv("train.csv")  # Read CSV file containing training data

# Separate features and target
X = df.drop("target", axis=1)  # Create feature matrix by removing target column
y = df["target"]  # Extract target variable column

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns  # Get names of all text/string columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns  # Get names of all numerical columns

# One-hot encoding for categorical columns
preprocessor = ColumnTransformer(  # Create preprocessing pipeline
    transformers=[  # Define transformations for different column types
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),  # One-hot encode categorical columns
        ("num", "passthrough", numerical_cols)  # Keep numerical columns as-is
    ]
)

# Apply encoding
X_encoded = preprocessor.fit_transform(X)  # Learn encoding and transform all features

# Split into train and test data
X_train, X_test, y_train, y_test = train_test_split(  # Split data into training and testing sets
    X_encoded, y, test_size=0.2, random_state=42  # 80% train, 20% test with fixed seed
)

# LOGISTIC REGRESSION USING SGD


model = SGDClassifier(  # Create Stochastic Gradient Descent classifier for logistic regression
    loss="log_loss",  # Use logistic loss function (for logistic regression)
    max_iter=1000,  # Maximum number of training iterations
    alpha=0.01,  # Regularization strength (higher = more regularization)
    penalty="l2",  # L2 regularization (ridge)
    random_state=42  # Fixed seed for reproducibility
)

# Train model
model.fit(X_train, y_train.values.flatten())  # Train model on training data (flatten converts to 1D array)

# Prediction
y_pred = model.predict(X_test)  # Make predictions on test data

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))  # Print percentage of correct predictions
print("ROC AUC:", roc_auc_score(y_test, y_pred))  # Print area under ROC curve


# ==============================
# FEATURE IMPORTANCE
# ==============================

# Get feature names after encoding
feature_names = preprocessor.get_feature_names_out()  # Get names of all features after preprocessing

# Convert coefficients to 1D
coef = model.coef_.flatten()  # Extract model coefficients and flatten to 1D array

# 10 least important features
print("10 Least Important Features:")  # Features with smallest absolute coefficients
print(feature_names[np.argsort(np.abs(coef))[:10]])  # Sort by absolute value and get first 10

# 10 most important features
print("10 Most Important Features:")  # Features with largest absolute coefficients
print(feature_names[np.argsort(np.abs(coef))[-10:]])  # Sort by absolute value and get last 10


# ==============================
# ONLINE LEARNING WITH PARTIAL FIT
# ==============================

online_model = SGDClassifier(  # Create new SGD classifier for online learning
    loss="log_loss",  # Logistic loss function
    learning_rate="constant",  # Use constant learning rate (doesn't decrease over time)
    eta0=0.01,  # Initial learning rate value
    penalty="l2",  # L2 regularization
    random_state=42  # Fixed seed for reproducibility
)

classes = np.unique(y_train.values.flatten())  # Get unique class labels from training data
batch_size = 1000  # Number of samples to process in each batch

# Train model sequentially in batches
for i in range(0, X_train.shape[0], batch_size):  # Loop through data in batches
    X_batch = X_train[i:i + batch_size]  # Extract batch of features
    y_batch = y_train.values.flatten()[i:i + batch_size]  # Extract batch of labels
    online_model.partial_fit(X_batch, y_batch, classes=classes)  # Update model incrementally with current batch

# Prediction
y_online_pred = online_model.predict(X_test)  # Make predictions using online-trained model

# Metrics
print("Online Learning Accuracy:", accuracy_score(y_test, y_online_pred))  # Print accuracy of online learning model
print("Online Learning ROC AUC:", roc_auc_score(y_test, y_online_pred))  # Print ROC-AUC of online learning model


# ==============================
# MULTICLASS LOGISTIC REGRESSION
# ==============================

# Generate synthetic multiclass dataset
X_mc, y_mc = make_classification(  # Create synthetic dataset for multiclass classification
    n_samples=1000,  # Total number of samples to generate
    n_features=20,  # Total number of features
    n_informative=15,  # Number of informative features (useful for prediction)
    n_redundant=5,  # Number of redundant features (linear combinations of informative features)
    n_classes=3,  # Number of classes (0, 1, 2)
    random_state=42  # Fixed seed for reproducibility
)

print(X_mc.shape, y_mc.shape)  # Print dimensions of synthetic dataset

# Train-test split
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(  # Split multiclass data
    X_mc, y_mc, test_size=0.2, random_state=42  # 80% train, 20% test
)

# Multiclass Logistic Regression model
mc_model = LogisticRegression(  # Create logistic regression for multiclass classification
    multi_class="multinomial",  # Use multinomial/softmax for multiclass (one-vs-rest alternative)
    solver="newton-cg",  # Optimization algorithm (Newton-CG method)
    max_iter=500  # Maximum number of iterations
)

# Train model
mc_model.fit(X_train_mc, y_train_mc)  # Train multiclass model

# Prediction
y_mc_pred = mc_model.predict(X_test_mc)  # Make multiclass predictions

# Metrics
print("Multiclass Accuracy:", accuracy_score(y_test_mc, y_mc_pred))  # Print overall accuracy
print("Multiclass F1 Score:", f1_score(y_test_mc, y_mc_pred, average="weighted"))  # Print weighted F1 score (accounts for class imbalance)